# Scraping Zara.com
Collecte des données produits via Selenium (Firefox headless) + regex sur le HTML rendu.

**Variables collectées :** nom, url, categorie, couleur, matiere, collection, pays, prix, promo, reference

## 0. Installation des dépendances

In [ ]:
# À lancer une seule fois si les packages ne sont pas installés
# !pip install selenium webdriver-manager beautifulsoup4 pandas

## 1. Imports & Configuration

In [1]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException
from webdriver_manager.firefox import GeckoDriverManager
import re
import time
import pandas as pd

In [2]:
# Colonnes du DataFrame final
VARIABLES = ["nom", "url", "categorie", "couleur", "matiere",
             "collection", "pays", "prix", "promo", "reference"]

df_zara = pd.DataFrame(columns=VARIABLES)
df_zara.head()

,nom,url,categorie,couleur,matiere,collection,pays,prix,promo,reference


In [3]:
# URLs de départ — pages liste de produits par collection
# (changer l'URL pour cibler une autre catégorie ou promotion)
URLS_LISTES = {
    "femme":   "https://www.zara.com/fr/fr/femme-nouveau-l1180.html",
    "homme":   "https://www.zara.com/fr/fr/homme-nouveau-l1179.html",
    "enfants": "https://www.zara.com/fr/fr/kids-fille-l6088.html",
}

PAYS = "France"

## 2. Lancement du navigateur (Firefox headless)

In [4]:
# headless=True = pas de fenêtre visible (recommandé pour du scraping en batch)
# Mettre headless=False si tu veux voir le navigateur s'ouvrir
HEADLESS = True

options = Options()
if HEADLESS:
    options.add_argument("--headless")

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)
print("Navigateur lancé.")

Navigateur lancé.


## 3. Extraction des URLs produits depuis une page liste

In [5]:
def get_product_urls(driver, listing_url, pause=8):
    """
    Charge une page liste Zara et extrait toutes les URLs produit.
    Les URLs produit Zara contiennent '-p' suivi de chiffres.
    """
    driver.get(listing_url)
    time.sleep(pause)

    # Scroll pour charger les produits en lazy-loading
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    links = driver.find_elements(By.TAG_NAME, "a")
    urls = set()
    for link in links:
        href = link.get_attribute("href")
        if href and "zara.com/fr/fr/" in href and re.search(r'-p\d+\.html', href):
            # Nettoyer les paramètres d'URL pour garder l'URL canonique
            clean = re.sub(r'\?.*$', '', href)
            urls.add(clean)

    return list(urls)

# Test sur femme
urls_femme = get_product_urls(driver, URLS_LISTES["femme"])
print(f"Femme : {len(urls_femme)} URLs trouvées")
urls_femme[:3]

Femme : 35 URLs trouvées


['https://www.zara.com/fr/fr/crcht-knt-16-p00021900.html',
 'https://www.zara.com/fr/fr/veste-courte-zw-collection-p02907085.html',
 'https://www.zara.com/fr/fr/jupe-midi-patchwork-zw-collection-limited-edition-p02790216.html']

## 4. Dictionnaire de couleurs (hex → nom)

In [6]:
# Palette Zara officielle (enrichie)
COULEURS_MAP = {
    "#000000": "Noir",
    "#1A1A1A": "Noir",
    "#FFFFFF": "Blanc",
    "#F5F5F5": "Blanc cassé",
    "#F0E7DF": "Beige",
    "#E8D8C4": "Beige",
    "#C8B89A": "Camel",
    "#8B6914": "Camel",
    "#4D5DEE": "Bleu",
    "#1B3A6B": "Bleu marine",
    "#87CEEB": "Bleu ciel",
    "#65C39F": "Vert",
    "#228B22": "Vert foncé",
    "#FE9459": "Orange",
    "#FFB4E2": "Rose",
    "#FF69B4": "Rose vif",
    "#D7DA73": "Jaune",
    "#FFD700": "Jaune",
    "#FF0000": "Rouge",
    "#8B0000": "Bordeaux",
    "#800080": "Violet",
    "#808080": "Gris",
    "#C0C0C0": "Gris clair",
    "#A52A2A": "Marron",
    "#D2691E": "Marron clair",
}

def hex_vers_couleur(hex_color):
    if not hex_color:
        return None
    return COULEURS_MAP.get(hex_color.upper(), f"Autre ({hex_color})")

## 5. Fonction de scraping d'un produit

In [7]:
def scrape_produit(driver, url, collection, pause=6):
    """
    Scrape les informations d'une page produit Zara.
    
    Retourne un dict avec :
      nom, url, categorie, couleur, matiere, collection, pays, prix, promo, reference
    """
    try:
        driver.get(url)
        time.sleep(pause)
        html = driver.page_source
    except WebDriverException as e:
        print(f"  [ERREUR WebDriver] {url} : {e}")
        return None

    # ── NOM ────────────────────────────────────────────────────────────────
    match = re.search(r'"name":"([^"]{3,100})"', html)
    nom = match.group(1) if match else None

    # ── RÉFÉRENCE ─────────────────────────────────────────────────────────
    match = re.search(r'"productRef":"([^"]+)"', html)
    reference = match.group(1) if match else None

    # ── CATÉGORIE (depuis le fil d'Ariane JSON-LD ou l'URL) ───────────────
    match = re.search(r'"breadcrumb".*?"name":"([^"]+)"', html, re.DOTALL)
    if match:
        categorie = match.group(1)
    else:
        # Fallback : extraire depuis l'URL  ex: /fr/fr/femme-robes-l1066
        match_url = re.search(r'/fr/fr/[a-z]+-([a-z-]+)-l\d+', url)
        categorie = match_url.group(1).replace("-", " ").title() if match_url else None

    # ── COULEUR ───────────────────────────────────────────────────────────
    match = re.search(r'"backgroundColor":"(#[A-Fa-f0-9]{6})"', html)
    hex_color = match.group(1) if match else None
    couleur = hex_vers_couleur(hex_color)

    # ── MATIÈRE PRINCIPALE ────────────────────────────────────────────────
    match = re.search(
        r'"material":"([^"]+)","percentage":"?([0-9]+)',
        html,
        flags=re.IGNORECASE
    )
    if match:
        matiere_principale = f"{match.group(2)}% {match.group(1)}"
    else:
        matiere_principale = None

    # ── PRIX ──────────────────────────────────────────────────────────────
    match_prix = re.search(r'"mainPrice":([0-9]+(?:\.[0-9]+)?)', html)
    prix = float(match_prix.group(1)) if match_prix else None

    # ── ANCIEN PRIX (si promo) ────────────────────────────────────────────
    match_old = re.search(r'"oldPrice":([0-9]+(?:\.[0-9]+)?)', html)
    old_price = float(match_old.group(1)) if match_old else None

    # ── PROMO (%) ─────────────────────────────────────────────────────────
    if old_price and prix and old_price > prix:
        promo = round(((old_price - prix) / old_price) * 100, 2)
    else:
        promo = 0

    # ── COLLECTION (depuis l'URL) ──────────────────────────────────────────
    # On utilise la valeur passée en paramètre (plus fiable que l'URL)

    return {
        "nom":        nom,
        "url":        url,
        "categorie":  categorie,
        "couleur":    couleur,
        "matiere":    matiere_principale,
        "collection": collection,
        "pays":       PAYS,
        "prix":       prix,
        "promo":      promo,
        "reference":  reference,
    }

# ── TEST sur une URL ─────────────────────────────────────────────────────
if urls_femme:
    test = scrape_produit(driver, urls_femme[0], "Femme")
    print(test)

{'nom': 'CRCHT KNT 16', 'url': 'https://www.zara.com/fr/fr/crcht-knt-16-p00021900.html', 'categorie': 'PNG', 'couleur': 'Beige', 'matiere': '76% coton', 'collection': 'Femme', 'pays': 'France', 'prix': 59.95, 'promo': 0, 'reference': '00021900-V2026'}


## 6. Scraping complet — toutes collections

In [8]:
def scrape_collection(driver, listing_url, collection_name, max_produits=None, pause=6):
    """
    Récupère toutes les URLs d'une page liste puis scrape chaque produit.
    max_produits : limite le nombre de produits (None = tous).
    """
    print(f"\n=== {collection_name.upper()} ===")
    urls = get_product_urls(driver, listing_url)
    print(f"  {len(urls)} URLs trouvées")

    if max_produits:
        urls = urls[:max_produits]

    rows = []
    for i, url in enumerate(urls, 1):
        print(f"  [{i}/{len(urls)}] {url}")
        row = scrape_produit(driver, url, collection_name, pause=pause)
        if row:
            rows.append(row)

    df = pd.DataFrame(rows, columns=VARIABLES)
    print(f"  → {len(df)} produits récupérés")
    return df

print("Fonction prête.")

Fonction prête.


In [9]:
# ── FEMME ────────────────────────────────────────────────────────────────
# max_produits=20 pour un test rapide ; mettre None pour tout scraper
df_femme = scrape_collection(driver, URLS_LISTES["femme"], "Femme", max_produits=20)
df_femme.head()


=== FEMME ===
  35 URLs trouvées
  [1/20] https://www.zara.com/fr/fr/crcht-knt-16-p00021900.html
  [2/20] https://www.zara.com/fr/fr/veste-courte-zw-collection-p02907085.html
  [3/20] https://www.zara.com/fr/fr/jupe-midi-patchwork-zw-collection-limited-edition-p02790216.html
  [4/20] https://www.zara.com/fr/fr/veste-cintree-a-brandebourgs-p04341764.html
  [5/20] https://www.zara.com/fr/fr/lunettes-de-soleil--il-de-chat-preppy-p02727009.html
  [6/20] https://www.zara.com/fr/fr/pantalon-avec-ceinture-en-lin-zw-collection-p01943904.html
  [7/20] https://www.zara.com/fr/fr/lunettes-de-soleil-maxi-masque-p02727011.html
  [8/20] https://www.zara.com/fr/fr/chemise-cintree-froncee-p04661097.html
  [9/20] https://www.zara.com/fr/fr/short-mini-denim-trf-etoiles-p06929004.html
  [10/20] https://www.zara.com/fr/fr/blazer-brandenbourgs-zw-collection-p02783617.html
  [11/20] https://www.zara.com/fr/fr/short-a-rayures-et-pinces-zw-collection-p01551355.html
  [12/20] https://www.zara.com/fr/fr/top-a-

,nom,url,categorie,couleur,matiere,collection,pays,prix,promo,reference
0,CRCHT KNT 16,https://www.zara.com/fr/fr/crcht-knt-16-p00021...,PNG,Beige,76% coton,Femme,France,59.95,0,00021900-V2026
1,VESTE COURTE ZW COLLECTION,https://www.zara.com/fr/fr/veste-courte-zw-col...,PNG,Beige,72% polyester,Femme,France,69.95,0,02907085-V2026
2,JUPE MIDI PATCHWORK ZW COLLECTION ÉDITION LIMITÉE,https://www.zara.com/fr/fr/jupe-midi-patchwork...,PNG,Beige,98% coton,Femme,France,99.95,0,02790216-V2026
3,BLOUSON AJUSTÉ À BRANDEBOURGS,https://www.zara.com/fr/fr/veste-cintree-a-bra...,PNG,Beige,100% coton,Femme,France,59.95,0,04341764-V2026
4,LUNETTES DE SOLEIL ŒIL-DE-CHAT PREPPY,https://www.zara.com/fr/fr/lunettes-de-soleil-...,PNG,Beige,81% diacétate de cellulose,Femme,France,45.95,0,02727009-V2026


In [11]:
# ── HOMME ────────────────────────────────────────────────────────────────
df_homme = scrape_collection(driver, URLS_LISTES["homme"], "Homme", max_produits=20)
df_homme.head()


=== HOMME ===
  64 URLs trouvées
  [1/20] https://www.zara.com/fr/fr/t-shirt-a-manches-longues-imprime-agustina-shuan-p00085062.html
  [2/20] https://www.zara.com/fr/fr/chemise-oversize-a-carreaux-avec-poche-zw-collection-p01063262.html
  [3/20] https://www.zara.com/fr/fr/chemise-oversize-a-rayures-p01971054.html
  [4/20] https://www.zara.com/fr/fr/t-shirt-effet-delave-paula-grenouille-p00085069.html
  [5/20] https://www.zara.com/fr/fr/robe-kaftan-fleurs-limited-edition-p02114845.html
  [6/20] https://www.zara.com/fr/fr/chemise-oversize-a-n-ud-p02484268.html
  [7/20] https://www.zara.com/fr/fr/blouson-bomber-oversize-a-fermeture-eclair-p04813305.html
  [8/20] https://www.zara.com/fr/fr/t-shirt-imprime-effet-delave-manuja-waldia-p06050073.html
  [9/20] https://www.zara.com/fr/fr/chemise-satinee-oversize-p08372119.html
  [10/20] https://www.zara.com/fr/fr/blazer-oversize-maxi-poches-p01255794.html
  [11/20] https://www.zara.com/fr/fr/t-shirt-manches-courtes-imprime-agustina-shuan-p00085

,nom,url,categorie,couleur,matiere,collection,pays,prix,promo,reference
0,T-SHIRT À MANCHES LONGUES IMPRIMÉ AGUSTINA SHUAN,https://www.zara.com/fr/fr/t-shirt-a-manches-l...,PNG,Beige,100% coton,Homme,France,22.95,0,00085062-V2026
1,CHEMISE OVERSIZE À CARREAUX AVEC POCHE ZW COLL...,https://www.zara.com/fr/fr/chemise-oversize-a-...,PNG,Beige,100% coton,Homme,France,39.95,0,01063262-V2026
2,CHEMISE OVERSIZE À RAYURES,https://www.zara.com/fr/fr/chemise-oversize-a-...,PNG,Beige,86% modal,Homme,France,32.95,0,01971054-V2026
3,T-SHIRT EFFET DÉLAVÉ PAULA GRENOUILLE,https://www.zara.com/fr/fr/t-shirt-effet-delav...,PNG,Beige,100% coton,Homme,France,22.95,0,00085069-V2026
4,ROBE KAFTAN FLEURS LIMITED EDITION,https://www.zara.com/fr/fr/robe-kaftan-fleurs-...,PNG,Beige,100% coton,Homme,France,79.95,0,02114845-V2026


In [12]:
# ── ENFANTS ──────────────────────────────────────────────────────────────
df_enfants = scrape_collection(driver, URLS_LISTES["enfants"], "Enfants", max_produits=20)
df_enfants.head()


=== ENFANTS ===
  5 URLs trouvées
  [1/5] https://www.zara.com/fr/fr/sweat-a-inscription-foil-p05644032.html
  [2/5] https://www.zara.com/fr/fr/sweat-broderie-fleurs-p01165038.html
  [3/5] https://www.zara.com/fr/fr/sweat-a-inscription-brillante-p05941003.html
  [4/5] https://www.zara.com/fr/fr/sweat-imprime-cortado-p00085054.html
  [5/5] https://www.zara.com/fr/fr/sweat-a-inscription-p03253019.html
  → 5 produits récupérés


,nom,url,categorie,couleur,matiere,collection,pays,prix,promo,reference
0,SWEAT À INSCRIPTION FOIL,https://www.zara.com/fr/fr/sweat-a-inscription...,PNG,Beige,97% coton,Enfants,France,29.95,0,05644032-V2026
1,SWEAT BRODERIE FLEURS,https://www.zara.com/fr/fr/sweat-broderie-fleu...,PNG,Beige,100% coton,Enfants,France,32.95,0,01165038-V2026
2,SWEAT À INSCRIPTION BRILLANTE,https://www.zara.com/fr/fr/sweat-a-inscription...,PNG,Beige,100% coton,Enfants,France,35.95,0,05941003-V2026
3,SWEAT IMPRIMÉ CORTADO,https://www.zara.com/fr/fr/sweat-imprime-corta...,PNG,Beige,65% coton,Enfants,France,35.95,0,00085054-V2026
4,SWEAT À INSCRIPTION,https://www.zara.com/fr/fr/sweat-a-inscription...,PNG,Beige,100% coton,Enfants,France,19.95,0,03253019-V2026


In [13]:
# Fermer le navigateur proprement
driver.quit()
print("Navigateur fermé.")

Navigateur fermé.


In [16]:
df_total = pd.concat([df_femme, df_homme, df_enfants], ignore_index=True)
df_total.head(10)
# Export CSV
df_total.to_csv("zara_data.csv", index=False, encoding="utf-8-sig")
print("Fichier 'zara_data.csv' exporté.")

Fichier 'zara_data.csv' exporté.


## 7. Fusion, nettoyage & export

In [ ]:
df_total = pd.concat([df_femme, df_homme, df_enfants], ignore_index=True)

# Nettoyage types
df_total["prix"]  = pd.to_numeric(df_total["prix"],  errors="coerce")
df_total["promo"] = pd.to_numeric(df_total["promo"], errors="coerce").fillna(0)

# Suppression des doublons sur la référence
df_total = df_total.drop_duplicates(subset="reference")

print(f"Total : {len(df_total)} produits")
print(f"\nRépartition par collection :")
print(df_total["collection"].value_counts())
print(f"\nProduits en promo : {(df_total['promo'] > 0).sum()}")
print(f"\nValeurs manquantes :")
print(df_total.isnull().sum())
print(f"\nStats prix & promo :")
print(df_total[["prix", "promo"]].describe())

In [ ]:
df_total.head(10)

In [ ]:
# Export CSV
df_total.to_csv("zara_data.csv", index=False, encoding="utf-8-sig")
print("Fichier 'zara_data.csv' exporté.")

# Exports séparés (optionnel)
# df_femme.to_csv("zara_femme.csv",   index=False, encoding="utf-8-sig")
# df_homme.to_csv("zara_homme.csv",   index=False, encoding="utf-8-sig")
# df_enfants.to_csv("zara_enfants.csv", index=False, encoding="utf-8-sig")